# Cattle Re-ID — Etapa 2, Fase 6: Harness de re-ID + gap crudo hocico→cara

Con el encoder source de la Fase 5 (`cmpd300_source.pt`), acá:
1. **Sanity intra-CMPD300** (plomería): Rank-1/mAP sobre las propias vacas de CMPD300. Como el
   encoder las vio en el entrenamiento hay *leakage* → el número NO es reportable, solo valida
   que el harness funciona (tiene que dar Rank-1 **alto**).
2. **Gap crudo hocico→cara** (reportable): Rank-1/mAP sobre las caras de Ahmed, identidades
   nuevas, **sin adaptar**. Es el punto de partida que la domain adaptation tiene que mejorar.

### Prerrequisitos
- Commitear al repo (fork, `rama_trini1`) los archivos nuevos: `src/reid/*` y `scripts/06_eval_reid.py`.
- En Drive `MyDrive/cattle_reid/`: `CMPD300_Baseline.zip`, `dataset_caras_bovinos.zip`, y el
  `cmpd300_source.pt` (ya persistido en `outputs/checkpoints/` desde la Fase 5).

## 0. GPU

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'Sin GPU: Entorno de ejecución -> Cambiar tipo -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')
assert DRIVE_ROOT.is_dir(), f'No existe {DRIVE_ROOT}.'
print('DRIVE_ROOT:', DRIVE_ROOT)

## 2. Traer el código (clon limpio de tu fork, rama_trini1)

In [ ]:
import os, shutil
os.chdir('/content')
REPO_URL = 'https://github.com/trinidadpujol/tp-final-vision2-Pujol-Vitale.git'
REPO_DIR = '/content/tp-final-vision2-Pujol-Vitale'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -b rama_trini1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

## 3. Verificar código de Fase 6 + checkpoint de Fase 5

In [ ]:
!python scripts/06_eval_reid.py \
    --source-dir "{SOURCE_DIR}" \
    --target-dir "{AHMED_ROOT}" \
    --by-session \
    --compare-imagenet

## 4. Persistir outputs en Drive (symlink) — trae el cmpd300_source.pt de la Fase 5

In [ ]:
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
for sub in ('checkpoints', 'logs', 'results'):
    drive_sub = DRIVE_OUTPUTS / sub; drive_sub.mkdir(parents=True, exist_ok=True)
    local_sub = Path(REPO_DIR) / 'outputs' / sub
    if local_sub.exists() and not local_sub.is_symlink():
        shutil.rmtree(local_sub)
    if not local_sub.exists():
        os.symlink(drive_sub, local_sub, target_is_directory=True)
    print(f'outputs/{sub} -> {drive_sub}')
assert (Path(REPO_DIR)/'outputs/checkpoints/cmpd300_source.pt').is_file(), 'no veo el checkpoint vía symlink'
print('checkpoint accesible ✅')

## 5. CMPD300 desde Drive (para el sanity)

Dataset chico; se extrae completo a disco local.

In [ ]:
IMG_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp'}
import zipfile
CMPD_ZIP = DRIVE_ROOT / 'CMPD300_Baseline.zip'
assert CMPD_ZIP.is_file(), f'Falta {CMPD_ZIP}.'
CMPD_LOCAL = Path('/content/cmpd300')
if not CMPD_LOCAL.exists():
    CMPD_LOCAL.mkdir(parents=True)
    !unzip -q "{CMPD_ZIP}" -d "{CMPD_LOCAL}"
cand = [CMPD_LOCAL, CMPD_LOCAL/'Baseline'] + [p for p in CMPD_LOCAL.iterdir() if p.is_dir()]
CMPD_DIR = next((p for p in cand if (p/'train').is_dir()), None)
assert CMPD_DIR, 'No encuentro train/ en el zip de CMPD300.'
SOURCE_DIR = CMPD_DIR / 'train'
print('SOURCE_DIR (sanity):', SOURCE_DIR)

## 6. Caras de Ahmed desde Drive — SUBSET (para el gap)

Ahmed pesa 13,9 GB: **no lo extraemos entero**. Leemos el índice del zip y extraemos solo un
subconjunto de individuos (K_IDS) con hasta MAX_PER_ID imágenes cada uno — suficiente para una
primera medición del gap, y rápido. Después se puede escalar.

In [ ]:
import random
FACES_ZIP = DRIVE_ROOT / 'dataset_caras_bovinos.zip'
assert FACES_ZIP.is_file(), f'Falta {FACES_ZIP}.'
K_IDS = 150          # individuos a muestrear
MAX_PER_ID = 24      # ~8 sesiones de 3 fotos por individuo      # imágenes por individuo a extraer

AHMED_ROOT = Path('/content/ahmed_subset')
if AHMED_ROOT.exists():
    shutil.rmtree(AHMED_ROOT)
AHMED_ROOT.mkdir(parents=True)

with zipfile.ZipFile(FACES_ZIP) as z:
    members = [m for m in z.namelist()
               if not m.endswith('/') and Path(m).suffix.lower() in IMG_EXTS]
    by_id = {}
    for m in members:
        by_id.setdefault(Path(m).parent.as_posix(), []).append(m)
    ids = list(by_id); random.Random(0).shuffle(ids); ids = ids[:K_IDS]
    n_imgs = 0
    for ind in ids:
        idname = Path(ind).name
        (AHMED_ROOT / idname).mkdir(exist_ok=True)
        for m in by_id[ind][:MAX_PER_ID]:
            (AHMED_ROOT / idname / Path(m).name).write_bytes(z.read(m))
            n_imgs += 1
print(f'Ahmed subset: {len(ids)} individuos, {n_imgs} imágenes en {AHMED_ROOT}')

## 7. Correr la Fase 6 (sanity + gap)

In [ ]:
!python scripts/06_eval_reid.py \
    --source-dir "{SOURCE_DIR}" \
    --target-dir "{AHMED_ROOT}" \
    --by-session \
    --compare-imagenet

## 8. Resultados

In [ ]:
import json
p = Path('outputs/results/06_reid_summary.json')
print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False))

## Cómo leer esto (split POR SESIÓN)

Ahora gallery y probe se separan por **sesión** (el timestamp del nombre): el probe se matchea
contra fotos de OTRA toma del mismo individuo, no contra su gemela de la misma ráfaga.

- **Sanity CMPD300**: Rank-1 alto → harness OK.
- **encoder de hocico vs ImageNet PURO** (mismo split honesto): si tu encoder ahora le **saca
  ventaja** a ImageNet → hay reconocimiento de hocico real, el número sirve como punto de partida
  para el DA. Si **siguen empatados** aun separando por sesión → la variación entre sesiones no
  alcanza (fotos demasiado parecidas), y es tema para Gastón: otro protocolo u otro target.